In [68]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')# 
dataset.fetch_dataset(name = "THUCNews_small", dataset_path = "/hongyi/stream/dataset/paper_data", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)

2025-03-13 17:49:38.643 | INFO     | stream_topic.utils.dataset:fetch_dataset:119 - Fetching dataset: THUCNews_small
2025-03-13 17:49:38.649 | INFO     | stream_topic.utils.dataset:fetch_dataset:129 - Fetching dataset from local path
Preprocessing documents:  43%|██████████▎             | 4194/9800 [00:50<00:48, 115.00it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x76ff01b29c90>>
Traceback (most recent call last):
  File "/hongyi/anaconda3/envs/mystream/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Preprocessing documents:  48%|███████████▉             | 4662/9800 [00:53<00:59, 86.76it/s]


KeyboardInterrupt: 

In [3]:
all_tokens = []
for tokens_list in dataset.dataframe["tokens"]: 
    all_tokens.extend(tokens_list) 
unique_tokens = set(all_tokens)
vocab_size = len(unique_tokens)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 145839


In [4]:
word_counts = dataset.dataframe['tokens'].apply(len)
stats = word_counts.describe() 
print("分词统计信息：")
print(stats)

分词统计信息：
count    19600.000000
mean       284.441837
std        316.581277
min          4.000000
25%        108.000000
50%        205.000000
75%        367.000000
max      13823.000000
Name: tokens, dtype: float64


In [ ]:
model=KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1/",stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')
model.fit(dataset,n_topics=14)
# model = NMFTM(stopwords_path = '/hongyi/stream/stopwords/scu_stopwords.txt')# 
# model.fit(dataset)#

topics = model.get_topics()
print(topics)

In [34]:
import json 
import pandas as pd 
# 打开并读取 JSON 文件 
with open('/hongyi/stream/dataset/CMtMedQA.json',  'r', encoding='utf-8') as file:
    data = json.load(file) 
 
# 将 JSON 数据转换为 DataFrame 
df = pd.DataFrame(data)
def flatten_history(history_list):
    if not isinstance(history_list, list) or len(history_list) == 0:
        return ''
    text_lines = []
    for conversation in history_list:
        for text in conversation:
            if isinstance(text, str):
                text_lines.append(text) 
    return '\n'.join(text_lines)
df['text'] = df['history'].apply(flatten_history)
df2 = df[['cate1','text']]
selected_categories = ['五官科', '传染病科','儿科','内科', '外科', '妇产科','男科', '皮肤科','心理科', '肿瘤科']
df3 = df2[df2['cate1'].isin(selected_categories)]
df3 = df3.rename(columns={'cate1':  'labels'}).reindex(columns=['text', 'labels'])
mask = df3['text'].str.strip()  == ''
df3 = df3[~mask].reset_index(drop=True)
df3 = df3.reset_index(drop=True) 
df3.to_csv('/hongyi/stream/dataset/paper_data/CMtMedQA_ten.txt',  sep='\t', index=False, encoding='utf-8')
sampled_df = df3.groupby('labels',  group_keys=False).apply(
    lambda x: x[['text', 'labels']].sample(min(len(x), 1000))
)
sampled_df = sampled_df.reset_index(drop=True) 
sampled_df.to_csv('/hongyi/stream/dataset/paper_data/CMtMedQA_small.txt',  sep='\t', index=False, encoding='utf-8')

In [164]:
import pandas as pd

# 读取文件并加载为 DataFrame
df = pd.read_csv('/hongyi/stream/dataset/paper_data/fudan_cleaned.txt', sep='\t', header=None, names=['text','labels'])
# df = pd.read_csv('/hongyi/stream/dataset/paper_data/Toutiao_long.txt',  sep='\t', encoding='utf-8')
# 查看结果
print(df.head())

                                                text    labels
0  江泽民总书记最近多次指出：“必须高度重视党的思想政治建设，大力提高干部和党员队伍的思想政治素...  Politics
1  巴巴多斯总理抵京开始访华 新华社北京５月９日电应国务院总理李鹏的邀请，巴          ...  Politics
2  新华社北京7月10日电  解放军总政治部、中央军委纪委最近发出通知，向全军转发了兰州军区某团...  Politics
3  江泽民同志在党的十五大报告中明确指出：“在社会主义初级阶段，围绕发展社会生产力这个根本任务，...  Politics
4  〔分类号〕B03    文献标识码：A  文章编号：1003—5281 （2000）03—0...  Politics


In [165]:
word_counts = df['text'].apply(len)
stats = word_counts.describe() 
print("分词统计信息：")
print(stats)

分词统计信息：
count    19614.000000
mean      5458.763026
std       3955.623448
min          5.000000
25%       2507.750000
50%       5130.000000
75%       7531.500000
max      47480.000000
Name: text, dtype: float64


In [166]:
mask = (word_counts >= 50) & (word_counts <= 5000)
new_df = df[mask].copy()

In [167]:
word_counts = new_df['text'].apply(len)
stats = word_counts.describe() 
print("分词统计信息：")
print(stats)

分词统计信息：
count    9540.000000
mean     2379.931132
std      1579.848780
min        51.000000
25%       737.250000
50%      2388.000000
75%      3839.000000
max      5000.000000
Name: text, dtype: float64


In [168]:
new_df.to_csv('/hongyi/stream/dataset/paper_data/fudan_short.txt',  sep='\t', index=False, encoding='utf-8')

In [162]:
sampled_df = df.groupby('labels',  group_keys=False).apply(
    lambda x: x[['text', 'labels']].sample(min(len(x), 1400))
)
sampled_df = sampled_df.reset_index(drop=True) 
sampled_df
sampled_df.to_csv('/hongyi/stream/dataset/paper_data/Toutiao_small.txt',  sep='\t', index=False, encoding='utf-8')

/tmp/ipykernel_817402/1402188512.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('labels',  group_keys=False).apply(


In [107]:
sampled_df

,text,labels
0,张亚东：世锦赛超奥运没问题 中国游泳已到腾飞时 新浪体育讯 4月12日晚，中国泳军在为期...,体育
1,中国军团14大项0金进账 三项亚运金牌数少于奥运 新浪体育讯 中国军团在本届亚运会上共夺...,体育
2,本菲卡觅得科恩特朗替身 宣布与西班牙国脚签约两年 新浪体育讯 北京时间7月21日消息，...,体育
3,命脉人物称伊布难离国米 防线大将告别蓝黑军提前宣布 新浪体育讯 伊布的去留风波炒了几天，...,体育
4,无唐江苏谁是真核心 胡雪峰得分之外还有可怕之处 本报记者阿笨报道 一直以来，胡雪峰都是球...,体育
...,...,...
13995,指基发行退潮 ETF逆势井喷 □本报记者 余? 北京报道 经历了2009年的发行“大...,财经
13996,500万吨最低价小麦入市遇冷 稳定麦价效果初现 胡军华 因终端面粉、麸皮等需求不旺，...,财经
13997,上期所输出中国标准见成效 已注册的境外品牌铜升至25个 证券时报记者 游 石 本...,财经
13998,金价距离历史高点7% 多数投资者短线看空黄金 自从09年12月份创出历史纪录后，黄金价格...,财经


In [108]:
word_counts = sampled_df['text'].apply(len)
stats = word_counts.describe() 
print("分词统计信息：")
print(stats)

分词统计信息：
count    14000.000000
mean       816.201643
std        619.065411
min         50.000000
25%        347.000000
50%        653.000000
75%       1131.000000
max       2998.000000
Name: text, dtype: float64


In [8]:
import pandas as pd
df = pd.read_csv('/hongyi/stream/dataset/paper_data/fudan_short.txt',  sep='\t', encoding='utf-8')
df

,text,labels
0,江泽民总书记最近多次指出：“必须高度重视党的思想政治建设，大力提高干部和党员队伍的思想政治素...,Politics
1,巴巴多斯总理抵京开始访华 新华社北京５月９日电应国务院总理李鹏的邀请，巴 ...,Politics
2,新华社北京7月10日电 解放军总政治部、中央军委纪委最近发出通知，向全军转发了兰州军区某团...,Politics
3,自去年9月布莱尔和克林顿等西方政要推出“第三条道路”以来，这一政治思潮很快风靡全球，在世界上...,Politics
4,朴成哲访问安哥拉 新华社罗安达５月４日电朝鲜民主主义人民共和国副 ...,Politics
...,...,...
9535,（本报记者 龚永泉 杨明方） 压题照片：熊猫电子集团公司彩电主板插件机生产线。（...,Electronics
9536,本报讯 记者赵志文报道：“长虹”ＩＳＯ９００１颁证暨新品展示会日前在北京 举行。四川长虹电...,Electronics
9537,本报讯 记者萧体焕报道：’９６北京电子商务国际论坛近日在北京举行。国务院 副总理邹家华出席...,Electronics
9538,（本报记者费伟伟） 赛格集团名气不小。 赛格成立于１９８６年，是全国最早的企...,Electronics


In [13]:
import numpy as np
len(np.unique(df['labels']))

20

In [9]:
#本段落用时9min
from stream_topic.utils.dataset import TMDataset

# 创建 TMDataset 实例
dataset = TMDataset()#language="zh-cn", stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt'

# 假设你有一个名为 df 的 DataFrame，包含你的数据
# df = pd.DataFrame(...)

# 数据集名称
dataset_name = "Fudan"

# 指定保存数据集的目录
save_dir = "/hongyi/stream/dataset/paper_data"
# df = df.iloc[0:100]
# 调用 create_load_save_dataset 方法来存储数据集
dataset.create_load_save_dataset(
    data=df,
    dataset_name=dataset_name,                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         
    save_dir=save_dir,
    doc_column="text",  # 假设 DataFrame 中包含文本的列名为 "text_column"
    label_column="labels",  # 假设 DataFrame 中包含标签的列名为 "label_column"
    language = "chinese",
    stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt',
    min_word_length = 1
)

# 打印保存的数据集信息
print(dataset.dataframe.head())

Preprocessing documents: 100%|█████████████████████████| 9540/9540 [02:19<00:00, 68.33it/s]
2025-03-13 20:32:14.573 | INFO     | stream_topic.utils.dataset:create_load_save_dataset:238 - Dataset saved to /hongyi/stream/dataset/paper_data/Fudan.parquet
2025-03-13 20:32:14.611 | INFO     | stream_topic.utils.dataset:create_load_save_dataset:255 - Dataset info saved to /hongyi/stream/dataset/paper_data/Fudan_info.pkl


  language                                 stopwords_path  min_word_length  \
0  chinese  /hongyi/stream/stopwords/common_stopwords.txt                1   
1  chinese  /hongyi/stream/stopwords/common_stopwords.txt                1   
2  chinese  /hongyi/stream/stopwords/common_stopwords.txt                1   
3  chinese  /hongyi/stream/stopwords/common_stopwords.txt                1   
4  chinese  /hongyi/stream/stopwords/common_stopwords.txt                1   

                                                text    labels  
0  指出 党 思想 政治 建设 干部 思想 政治素质 切实 好 世界观 人生观 问题 党 一定 ...  Politics  
1  巴巴多斯 总理 北京 应 总理 巴巴多斯 总理 桑迪福德 一行 北京 中国 进行 桑迪福德 ...  Politics  
2  新华社 北京 月 日电 最近 转发 兰州军区 某团 党委 真抓实干 锲而不舍 加强 党风廉政...  Politics  
3  月 布莱尔 克林顿 西方 推出 第三条 道路 这一 政治思潮 产生 治国 这一 思潮 左右翼...  Politics  
4  朴成哲 访问 安哥拉 新华社 罗安达 月 日电 副 朴成哲 结束 安哥拉 天 正式 访问 后...  Politics  


In [6]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')# 
# dataset.fetch_dataset(name = "cnews_test", dataset_path = "/hongyi/stream/dataset", source = 'local')
dataset.fetch_dataset(name = "my_dataset", dataset_path = "/hongyi/stream/dataset", source = 'local')
dataset.preprocess(model_type="NMFTM", min_word_length = 1)
#本段落用时4h
# model = KmeansTM(embedding_model_name="/hongyi/stream/sentence-transformers/Conan-embedding-v1/",stopwords_path = '/hongyi/stream/stopwords/baidu_stopwords.txt')#
# model.fit(dataset,n_topics=2)
# model = CBC(language="chinese",stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt',threshold=0.8)#,
# model.fit(dataset)

# topics = model.get_topics()
# print(topics)

2025-03-14 10:35:39.354 | INFO     | stream_topic.utils.dataset:fetch_dataset:119 - Fetching dataset: my_dataset
2025-03-14 10:35:39.357 | INFO     | stream_topic.utils.dataset:fetch_dataset:129 - Fetching dataset from local path
Preprocessing documents: 100%|████████████████████████████████████████████████████████████████████████████████| 2500/2500 [00:10<00:00, 230.97it/s]


In [13]:
dataset.dataframe['text']

0       出局 却 里 弗斯 继续 来说 却 时 波士顿 凯尔特人 热火 出局 波士顿 赛季 弗斯 却...
1       魔术 老鹰 主帅 晋级 魔术 老鹰队 两队 总比分 老鹰 魔术 老鹰 主场 大比分 上 一场...
2       科比 对抗 新浪 体育讯 时间 洛杉矶 湖人队 利用 好 身高 优势 争取 再 一次 西部 ...
3       良性 竞争 仇恨 暴力 恤 新浪 体育讯 北京 时间 月 日 第二轮 湖人 小牛 系列赛 球...
4       湖人 小牛 前瞻 科比 横扫 禅师 新浪 体育讯 北京 时间 月 日 消息 菲尔 杰克逊 教...
                              ...                        
2495    鹏华 沪 深 指数 基金 日 成立 首募 近 亿元 全景网 月 基金 公告 称 旗下 鹏华 ...
2496    封闭式 基金 昨日 分红 本报讯 记者 昨日 基金安 信 发布公告 分红 成 今年 第一 只...
2497    建信 获 值得 托付 基金 公司 奖 年度 暨 第二届 中国 理财 获奖 名单 近日 出炉 ...
2498    上 投 摩根 制度 带来 稳定 收益 上 投 摩根 策略 研究员 近期 欧洲 参加 培训 一...
2499    持 观望 态度 新 基金 建仓 普遍 谨慎 徐婧婧 今年以来 股 市场 强势 反弹 今年 成...
Name: text, Length: 2500, dtype: object

In [ ]:
from stream_topic.models import KmeansTM,BERTopicTM,CBC,DCTE,NMFTM,SOMTM,CEDC,ETM,LDA,ProdLDA,SOMTM,NSTM,WordCluTM,CTM,TNTM,NeuralLDA,CTMNeg
from stream_topic.utils import TMDataset
#本段落用时9min
dataset = TMDataset(language="chinese", stopwords_path = '/hongyi/stream/stopwords/common_stopwords.txt')# 
# dataset.fetch_dataset(name = "cnews_test", dataset_path = "/hongyi/stream/dataset", source = 'local')
dataset.fetch_dataset(name = "THUCNews", dataset_path = "/hongyi/stream/dataset/paper_data", source = 'local')
dataset.preprocess(model_type="KmeansTM", min_word_length = 1)

2025-03-14 10:46:37.821 | INFO     | stream_topic.utils.dataset:fetch_dataset:119 - Fetching dataset: THUCNews
2025-03-14 10:46:37.823 | INFO     | stream_topic.utils.dataset:fetch_dataset:129 - Fetching dataset from local path
